In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

I0000 00:00:1783920192.994317    3962 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1783920193.052111    3962 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1783920211.897072    3962 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
## Load the trained model, scalar pickle, onehot

model = load_model('model.h5')

## Load the encoder and scalar

with open('onehot_encoder_geo.pkl', 'rb') as file:
  label_encoder_geo=pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
  label_encoder_gender=pickle.load(file)

with open('scaler.pkl', 'rb') as file:
  scaler=pickle.load(file)

I0000 00:00:1783920224.854502    3962 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3617 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:02:00.0, compute capability: 8.6


In [3]:
## Example input data

input_data={
  'CreditScore': 600,
  'Geography' : 'France',
  'Gender' : 'Male',
  'Age' : 40,
  'Tenure' : 3,
  'Balance' : 60000,
  'NumOfProducts' : 2,
  'HasCrCard' : 1,
  'IsActiveMember' : 1,
  'EstimatedSalary' : 50000
}

In [4]:
## one-hot-encoding the 'Geography"

geo_encoded = label_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=label_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

/mnt/d/myProjects/endtoend-ann/.venv-linux/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [5]:
## First converting it into a dataframe

input_df = pd.DataFrame([input_data])
input_df


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [6]:
## Encode categorical variables only doing it for the Gender

input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [7]:
## Concetenation process of the one hot encoded data

input_df=pd.concat([input_df.drop("Geography", axis=1), geo_encoded_df], axis = 1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [8]:
## Scaling the input data

input_scaled=scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [9]:
## Final prediction churn

prediction = model.predict(input_scaled)
prediction

I0000 00:00:1783920226.420419    4210 service.cc:153] XLA service 0x7f973002d4c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1783920226.420452    4210 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3050 6GB Laptop GPU, Compute Capability 8.6 (Driver: 13.1.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1783920226.424748    4210 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1783920226.444113    4210 cuda_dnn.cc:461] Loaded cuDNN version 92400


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 536ms/step


I0000 00:00:1783920226.896725    4210 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


array([[0.06359422]], dtype=float32)

In [10]:
prediction_proba = prediction[0][0]

In [11]:
prediction_proba

0.06359422

In [12]:
if(prediction_proba > 0.5):
  print("Person is likely to churn")
else:
  print("Person is not likely to churn")

Person is not likely to churn
